# To Search Elasticsearch

This script is designed to be a template for elasticsearch instances

In [ ]:
import sys
sys.path.append('..')
from credentials import *

from elasticsearch_utils import *

import pandas as pd
import numpy as np
import re

import json

import random

from medcat.cat import CAT

data_path = "../data/"
raw_data_path = data_path+'raw_data/'

# Login and Initialise

In [ ]:
# --- Connect to Elasticsearch ---
es = connect_elasticsearch(hosts=hosts, username=username, password=password, api_key=True)

# Check the list of Indices and columns

In [ ]:
# --- Explore Indices ---
print("🔍 Available Indices:")
for i in list(es.indices.get_alias().keys()):
    print("-", i)

In [ ]:
# --- Explore Fields in a Given Index ---
index = [
    'noting'
    'letters'
    'notes'
        ]  # 🔧 Set your index name here

for idx in index:
    print(f"\n📑 Fields in index: {idx}")
    mapping = get_field_mapping(es, idx)
    for field in mapping[idx]["mappings"]["properties"]:
        print("-", field)

In [ ]:
inclusion_patients_df = pd.read_csv(os.path.join(data_path, "chronic_kidney_disease_refined_inclusion_patients.csv"))

inclusion_patients_df = inclusion_patients_df[inclusion_patients_df['addl_pat_flg']==1]

In [ ]:
patient_identifier1_number = []
for idx, row in enumerate(inclusion_patients_df['patient_identifier4']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            patient_identifier1_number.append(i)

In [ ]:
patient_identifier3_number = []
for idx, row in enumerate(inclusion_patients_df['patient_identifier3']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            try:
                patient_identifier3_number.append(int(i))
            except:
                pass

In [ ]:
patient_identifier2_number = []
for idx, row in enumerate(inclusion_patients_df['patient_identifier2']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            patient_identifier2_number.append(int(i))

In [ ]:
search_names = json.load(open(raw_data_path+'project_filter/20250711_cui_names_for_search.json'))

# Set parameters and define columns of interest

Select your fields and list in order of output columns

In [ ]:
# --- Define Query Parameters ---
columns = ['patient_identifier4', 'patient_identifier2', 'patient_identifier3',
           'document_Name', 'document_CreatedWhen', 'document_Content']  # 🔧 List of fields you'd like to return (or leave empty to return all)

## Build query

For further information on [how to build a query can be found here](https://www.elastic.co/guide/en/elasticsearch/reference/current/query-dsl.html)

Further information on [free text string queries can be found here](https://www.elastic.co/guide/en/elasticsearch/reference/current/query-dsl-simple-query-string-query.html)


# Search, Process, and Save

In [ ]:
cat = CAT.load_model_pack('../../../models/20241120_trained_ckd_model/medcat_model.zip')

In [ ]:
total_codes = len(search_names)

In [ ]:
docs_to_remove = ['Notes', 'Clinic Letters', 'Admin Letters']

In [ ]:
df = pd.DataFrame()
i = 1
results = 0
id_set = set()

for code, names in search_names.items():
    names_list = [f'"{name}"' for name in names]
    names_list = ' OR '.join(names_list)

    print(f'{code}: {cat.cdb.cui2preferred_name[code]}')
    
    query = {
        "from": 0,
        "size": 10000,
        "query": {
            "bool": {
                "must": [
                    {
                        "query_string": {
                            "query": f"document_Content:({names_list})",
                            "analyze_wildcard": True,
                            "time_zone": "Europe/London"
                        }
                    }
                ],
                "filter": [
                    {
                        "bool": {
                            "should": [
                                {"terms": {"patient_identifier1": patient_identifier4_number}},
                                {"terms": {"patient_identifier3": patient_identifier3_number}},
                                {"terms": {"patient_identifier2": patient_identifier2_number}}
                            ],
                            "minimum_should_match": 1,
                            "must_not": [
                                {"terms": {"document_Name.keyword": docs_to_remove}}
                            ]
                        }
                    }
                ]
            }
        }
    }

    temp_df = es_docs_to_df(es, index=index, query=query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(df['_id'].to_list())]
            temp_df = temp_df[~temp_df['document_Name'].isin(docs_to_remove)]
        except Exception as e:
            print(f"Deduplication error: {e}")
    
        df = pd.concat([df, temp_df], ignore_index=True)
        df = df.drop_duplicates()

    del temp_df
    print(f'{(i/total_codes)*100:.2f}% Complete ({i}/{total_codes})')
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {df.shape[0]:,}\n")

    i += 1

## Process

In [ ]:
# Whatever you want here
df.head()

## Save

In [ ]:
# --- Save Results ---
path_to_results = raw_data_path+'elasticsearch_search_hits'
if index[0] == 'noting':
    file_name = "20260202_clinical_noting_comorbidities_search.csv"
elif index[0] == 'letters':
    file_name = "20260202_epr_comorbidities_search.csv"
elif index[0] == 'notes':
    file_name = "20260202_epic_comorbidities_search.csv"
df.to_csv(f"{path_to_results}/{file_name}", index=False)
print("✅ Results saved.")

## Sandbox